# Klassifikation der Schmerzintensität t0-t5

In [2]:
from pathlib import Path
import numpy as np
from scripts.myml import loso_multiclass_nested_cv
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

### Daten laden

In [3]:
DATASET_PATH = Path("dataset/np-dataset")
X = np.load(DATASET_PATH / "X.npy")
X_catch22 = np.load(DATASET_PATH / "X_catch22.npy")
X_catch22_feature_names = np.load(DATASET_PATH / "feature_names.npy")
y_heater = np.load(DATASET_PATH / "y_heater.npy")
subjects = np.load(DATASET_PATH / "subjects.npy")
y = np.argmax(y_heater, axis=1)

In [4]:
print(f"X shape : {X_catch22.shape}")
print(f"y shape : {y.shape}   classes: {np.unique(y).tolist()}")
print(f"subjects : {np.unique(subjects).shape[0]} unique subjects")
print(f"Class counts : { {c: int((y==c).sum()) for c in range(6)} }")

X shape : (2495, 174)
y shape : (2495,)   classes: [0, 1, 2, 3, 4, 5]
subjects : 52 unique subjects
Class counts : {0: 416, 1: 416, 2: 416, 3: 415, 4: 416, 5: 416}


### Hyperparameter Grid

In [5]:
PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

INNER_N_SPLITS = 5
RANDOM_STATE = 42

# Klasifikation der Intensitäten t0-t5

### Random Forest vs Random Forest Regressor

In [6]:
# Define models
rf_classifier = RandomForestClassifier(random_state=RANDOM_STATE)
rf_regressor = RandomForestRegressor(random_state=RANDOM_STATE)
models = {
    "Random Forest Classifier": rf_classifier,
    "Random Forest Regressor": rf_regressor
}

# Define comparisons
model_types = {
    "Random Forest Classifier": "classifier",
    "Random Forest Regressor": "regressor"
}

# Train and evaluate models using LOSO nested cv
for name, model in models.items():

    metrics = loso_multiclass_nested_cv(
        X_catch22,
        y,
        subjects,
        model,
        PARAM_GRID,
        classes=np.unique(y),
        model_type=model_types[name]
    )

    print(f"\n{name}")
    print(f"    Accuracy : {metrics['accuracy']:.3f} ± {metrics['accuracy_std']:.3f}")
    print(f"    F1       : {metrics['f1']:.3f} ± {metrics['f1_std']:.3f}")
    print(f"    AUC      : {metrics['auc']:.3f} ± {metrics['auc_std']:.3f}")
    print(f"    MAE      : {metrics['mae']:.3f} ± {metrics['mae_std']:.3f}")
    print(f"    RMSE      : {metrics['rmse']:.3f} ± {metrics['rmse_std']:.3f}")


KeyboardInterrupt: 